--Prueba Commit

In [0]:
DESCRIBE KRAF_SUPABASE_catalog.public.transferencia;

In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.kraf_bronze;

In [0]:
CREATE TABLE IF NOT EXISTS workspace.kraf_bronze.transferencia_bronze AS
SELECT
    transferencia_id,
    cuenta_id,
    entidad_financiera_id,
    tipo_transferencia_id,
    monto,
    fecha_hora,
    cuenta_destino,
    referencia,
    canal,
    estado,
    current_timestamp() AS _ingested_at,
    'Supabase' AS _source
FROM KRAF_SUPABASE_catalog.public.transferencia
WHERE 1 = 0;

In [0]:
INSERT INTO workspace.kraf_bronze.transferencia_bronze
SELECT
    transferencia_id,
    cuenta_id,
    entidad_financiera_id,
    tipo_transferencia_id,
    monto,
    fecha_hora,
    cuenta_destino,
    referencia,
    canal,
    estado,
    current_timestamp() AS _ingested_at,
    'Supabase' AS _source
FROM KRAF_SUPABASE_catalog.public.transferencia;

In [0]:
SELECT COUNT(*) AS total_registros
FROM workspace.kraf_bronze.transferencia_bronze;

In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.kraf_silver;

In [0]:
CREATE TABLE IF NOT EXISTS workspace.kraf_silver.transferencia_silver AS
WITH datos_limpios AS (
    SELECT
        transferencia_id,
        cuenta_id,
        entidad_financiera_id,
        tipo_transferencia_id,
        monto,
        fecha_hora,
        cuenta_destino,
        referencia,
        canal,
        estado,
        _ingested_at,
        _source,
        ROW_NUMBER() OVER (
            PARTITION BY transferencia_id
            ORDER BY _ingested_at DESC
        ) AS fila
    FROM workspace.kraf_bronze.transferencia_bronze
    WHERE transferencia_id IS NOT NULL
      AND cuenta_id IS NOT NULL
      AND monto IS NOT NULL
      AND monto > 0
)
SELECT
    transferencia_id,
    cuenta_id,
    entidad_financiera_id,
    tipo_transferencia_id,
    monto,
    fecha_hora,
    cuenta_destino,
    referencia,
    canal,
    estado,
    _ingested_at,
    _source
FROM datos_limpios
WHERE fila = 1;

In [0]:
SELECT COUNT(*) AS total_registros
FROM workspace.kraf_silver.transferencia_silver;

In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.kraf_gold;

In [0]:
CREATE OR REPLACE TABLE workspace.kraf_gold.transferencias_atipicas AS
WITH transferencias_con_promedio AS (
    SELECT
        transferencia_id,
        cuenta_id,
        entidad_financiera_id,
        tipo_transferencia_id,
        monto,
        fecha_hora,
        cuenta_destino,
        referencia,
        canal,
        estado,
        AVG(monto) OVER (PARTITION BY canal) AS promedio_canal
    FROM workspace.kraf_silver.transferencia_silver
)
SELECT
    transferencia_id,
    cuenta_id,
    entidad_financiera_id,
    tipo_transferencia_id,
    monto,
    fecha_hora,
    cuenta_destino,
    referencia,
    canal,
    estado,
    ROUND(promedio_canal, 2) AS promedio_canal,
    ROUND(monto / promedio_canal, 2) AS veces_promedio,
    CASE
        WHEN monto > (promedio_canal * 2) THEN 'ATIPICA'
        ELSE 'NORMAL'
    END AS clasificacion
FROM transferencias_con_promedio;

In [0]:
SELECT
    clasificacion,
    COUNT(*) AS total
FROM workspace.kraf_gold.transferencias_atipicas
GROUP BY clasificacion
ORDER BY total DESC;

In [0]:
SELECT COUNT(*) AS total_registros
FROM workspace.kraf_gold.transferencias_atipicas;

In [0]:
MERGE INTO workspace.kraf_bronze.transferencia_bronze AS destino
USING KRAF_SUPABASE_catalog.public.transferencia AS origen
ON destino.transferencia_id = origen.transferencia_id

WHEN NOT MATCHED THEN
INSERT (
    transferencia_id,
    cuenta_id,
    entidad_financiera_id,
    tipo_transferencia_id,
    monto,
    fecha_hora,
    cuenta_destino,
    referencia,
    canal,
    estado,
    _ingested_at,
    _source
)
VALUES (
    origen.transferencia_id,
    origen.cuenta_id,
    origen.entidad_financiera_id,
    origen.tipo_transferencia_id,
    origen.monto,
    origen.fecha_hora,
    origen.cuenta_destino,
    origen.referencia,
    origen.canal,
    origen.estado,
    current_timestamp(),
    'Supabase'
);

In [0]:
SELECT COUNT(*) AS total_bronze
FROM workspace.kraf_bronze.transferencia_bronze;